# Lesson 2.3 — Prismatic Joints, Mixed Chains, the Full 6×n Jacobian
**Module 6 · Unit 2 · Lesson 7**

A prismatic joint contributes $[\mathbf z;\mathbf 0]$ — pure translation, **zero** angular part. We assemble a mixed revolute–prismatic chain, confirm the prismatic column has no angular component, and validate the full $6\times n$ Jacobian against finite differences. (Companion to the Jacobian Column Explorer demo.)

In [1]:
import numpy as np

def skew(v):
    v=np.asarray(v,float).ravel()
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])

def dh(theta,d,a,alpha):
    # one Denavit-Hartenberg link transform
    ct,st,ca,sa=np.cos(theta),np.sin(theta),np.cos(alpha),np.sin(alpha)
    return np.array([[ct,-st*ca, st*sa, a*ct],
                     [st, ct*ca,-ct*sa, a*st],
                     [0,  sa,    ca,    d   ],
                     [0,  0,     0,     1   ]])

def forward_chain(params,jtypes,q):
    # returns T_0^0..T_0^n ; revolute adds q to theta, prismatic adds q to d
    T=np.eye(4); Ts=[T.copy()]
    for i,(th0,d0,a,al) in enumerate(params):
        th,d=(th0+q[i],d0) if jtypes[i]=="R" else (th0,d0+q[i])
        T=T@dh(th,d,a,al); Ts.append(T.copy())
    return Ts

def geometric_jacobian(params,jtypes,q):
    # base-frame 6xn geometric Jacobian (D-057: [v; omega], linear on top)
    Ts=forward_chain(params,jtypes,q); o_n=Ts[-1][:3,3]; n=len(q); J=np.zeros((6,n))
    for i in range(n):
        z=Ts[i][:3,2]; o=Ts[i][:3,3]   # axis & origin of frame i (= z_{i-1}, o_{i-1})
        if jtypes[i]=="R": J[:3,i]=np.cross(z,o_n-o); J[3:,i]=z   # revolute: [z x (o_n-o); z]
        else:             J[:3,i]=z;                   J[3:,i]=0   # prismatic: [z; 0]
    return J

def fk_pose(params,jtypes,q):
    T=forward_chain(params,jtypes,q)[-1]; return T[:3,3], T[:3,:3]


## A mixed revolute–prismatic–revolute chain

In [2]:
checks=[]
params=[(0,0,0,np.pi/2),(0,0,0,0),(0,0,0.5,0)]; jtypes=["R","P","R"]
q=np.array([0.4,0.3,-0.5])
J=geometric_jacobian(params,jtypes,q)
print("6x3 Jacobian:\n",np.round(J,4))

6x3 Jacobian:
 [[ 0.1054  0.3894  0.2208]
 [ 0.521  -0.9211  0.0933]
 [-0.      0.      0.4388]
 [ 0.      0.      0.3894]
 [ 0.      0.     -0.9211]
 [ 1.      0.      0.    ]]


## The prismatic column has zero angular part (sliding makes no rotation)

In [3]:
pris_idx=[i for i,t in enumerate(jtypes) if t=="P"]
for i in pris_idx:
    print(f"column {i} angular block:",np.round(J[3:,i],6))
    checks.append(np.allclose(J[3:,i],0))

column 1 angular block: [0. 0. 0.]


## Full 6×n Jacobian matches finite differences

In [4]:
def fd_jac(params,jtypes,q,eps=1e-6):
    n=len(q); Jf=np.zeros((6,n))
    for i in range(n):
        e=np.zeros(n); e[i]=eps
        pp,Rp=fk_pose(params,jtypes,q+e); pm,Rm=fk_pose(params,jtypes,q-e)
        Jf[:3,i]=(pp-pm)/(2*eps); M=Rp@Rm.T; Jf[3:,i]=vee((M-M.T)/2)/(2*eps)
    return Jf
checks.append(np.allclose(J,fd_jac(params,jtypes,q),atol=1e-5))
print("geometric == finite-difference:",checks[-1])

geometric == finite-difference: True


In [5]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
